# Attention Position Encoding

How position information is encoded and used in attention:
positional bias, sensitivity, relative preferences, and encoding strength.

## Why This Matters

Attention position encoding reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_position_encoding import (
    positional_attention_bias, position_sensitivity,
    relative_position_preference, position_encoding_strength,
    position_encoding_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Positional Attention Bias

Correlation between attention and a purely positional (uniform causal) pattern.

In [ ]:
for layer in range(4):
    result = positional_attention_bias(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_positional_heads']} positional heads")
    for h in result['per_head']:
        flag = ' [POS]' if h['is_positional'] else ''
        print(f"    H{h['head']}: corr={h['positional_correlation']:.4f}{flag}")

## Position Sensitivity

Do attention patterns change with query position?

In [ ]:
for layer in range(4):
    result = position_sensitivity(model, tokens, layer=layer)
    print(f"  Layer {layer}: sensitivity={result['mean_sensitivity']:.2f}")
    for h in result['per_head']:
        flag = ' [SENSITIVE]' if h['is_position_sensitive'] else ''
        print(f"    H{h['head']}: sim={h['pattern_similarity']:.4f}{flag}")

## Relative Position Preference

Attention as a function of distance — recent vs distant tokens.

In [ ]:
for layer in range(4):
    for head in range(4):
        r = relative_position_preference(model, tokens, layer, head)
        flag = ' [RECENT]' if r['prefers_recent'] else ''
        print(f"  L{layer}H{head}: peak_dist={r['peak_distance']}{flag}")

## Position Encoding Strength

How distinct are positions in the residual stream?

In [ ]:
result = position_encoding_strength(model, tokens)
print(f"Most distinct layer: {result['most_distinct_layer']}")
for p in result['per_layer']:
    flag = ' [DISTINCT]' if p['position_distinct'] else ''
    print(f"  Layer {p['layer']}: sim={p['mean_position_similarity']:.4f}{flag}")

## Position Encoding Summary

In [ ]:
result = position_encoding_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: positional_heads={p['n_positional_heads']}, "
          f"pos_sim={p['position_similarity']:.4f}")